In [4]:
import pandas as pd
import numpy as np
import ast
import re
from sklearn.preprocessing import MultiLabelBinarizer

# --- 1. Hardcoded Configuration ---
FILE_MARCH = pd.read_csv("https://www.dropbox.com/scl/fi/dzdshl185gm7i0k959tbr/insideAirBnBdataset1.csv?rlkey=7ngn45e1042brxk7y4po3bgm0&st=o1equgch&dl=1")            # Dataset 1 (March)
FILE_JUNE  = pd.read_csv("https://www.dropbox.com/scl/fi/f3o8e3ne7uvsglfyjgeei/insideAirBnBdataset2.csv?rlkey=sz3z15ds9p57ahfgmh5vnl0dv&st=bt28midv&dl=1")          # Dataset 2 (June)
FILE_Q4    = "scraped_data.csv"  # Dataset 3 (Scraped Oct/Nov/Dec)

def clean_price(x):
    """Cleans '$1,200.00' and '€ ,212.0' into float numbers."""
    if pd.isna(x): return np.nan
    x = str(x)
    clean = re.sub(r'[^\d.]', '', x) 
    try: return float(clean)
    except ValueError: return np.nan

def clean_boolean(x):
    if pd.isna(x): return 0
    return 1 if str(x).lower().strip() in ['t', 'true', '1', 'yes'] else 0

def parse_amenities(x):
    """Parses lists of amenities safely."""
    if pd.isna(x): return []
    try:
        return x if isinstance(x, list) else ast.literal_eval(x)
    except:
        clean = str(x).replace('[', '').replace(']', '').replace('"', '').replace("'", "")
        return [i.strip() for i in clean.split(',') if i.strip()]

def merge_datasets():
    print("--- 1. Loading Datasets ---")
    
    # A. Load March (Base)
    try:
        df_march = FILE_MARCH.copy()
        df_march['month'] = 3  # Hardcode seasonality
        print(f"Loaded March: {df_march.shape}")
    except FileNotFoundError:
        print(f"Error: Could not find {FILE_MARCH}")
        return

    # B. Load June (Seasonal)
    try:
        df_june = FILE_JUNE.copy()
        df_june['month'] = 6   # Hardcode seasonality
        print(f"Loaded June:  {df_june.shape}")
    except FileNotFoundError:
        print(f"Error: Could not find {FILE_JUNE}")
        return

    # C. Load Q4 (Scraped)
    try:
        df_q4 = pd.read_csv(FILE_Q4)
        # Use existing q4_month column, or default to 10 (October) if missing
        if 'q4_month' in df_q4.columns:
            df_q4['month'] = pd.to_numeric(df_q4['q4_month'], errors='coerce')
        else:
            df_q4['month'] = 10 
        print(f"Loaded Q4:    {df_q4.shape}")
    except FileNotFoundError:
        print(f"Error: Could not find {FILE_Q4}")
        df_q4 = pd.DataFrame()

    print("\n--- 2. Normalizing Columns ---")

    # --- Prepare Historical Data (March & June) ---
    historical_dfs = [df_march, df_june]
    for df in historical_dfs:
        # Host Years
        df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
        df['host_years'] = (pd.Timestamp.now() - df['host_since']).dt.days / 365.0
        
        # Booleans
        df['superhost'] = df['host_is_superhost'].apply(clean_boolean)
        df['instant_bookable'] = df['instant_bookable'].apply(clean_boolean)
        
        # Room Type
        df['is_entire_home'] = df['room_type'].apply(lambda x: 1 if 'Entire' in str(x) else 0)

    # --- Prepare Scraped Data (Q4) ---
    if not df_q4.empty:
        # Host Years (from numeric column)
        if 'passportData.timeAsHost.years' in df_q4.columns:
            df_q4['host_years'] = pd.to_numeric(df_q4['passportData.timeAsHost.years'], errors='coerce')
        else:
            df_q4['host_years'] = 0

        # Booleans
        col_super = 'passportData.isSuperhost' if 'passportData.isSuperhost' in df_q4.columns else 'superhost'
        df_q4['superhost'] = df_q4.get(col_super, pd.Series(0)).apply(clean_boolean)
        df_q4['instant_bookable'] = 0 # Scraper didn't catch this

        # Room Type (from Title)
        if 'title' in df_q4.columns:
            df_q4['is_entire_home'] = df_q4['title'].apply(lambda x: 0 if 'Room' in str(x) else 1)
        else:
            df_q4['is_entire_home'] = 1

        # Rename to match Inside Airbnb
        rename_map = {
            'coordinates.latitude': 'latitude',
            'coordinates.longitud': 'longitude',
            'amenities_list': 'amenities',
            'rating.reviewCount': 'number_of_reviews',
            'rating.value': 'review_scores_rating',
        }
        df_q4.rename(columns=rename_map, inplace=True)

    # --- 3. Merge ---
    print("\n--- 3. Merging Everything ---")
    
    # Features we want
    cols = ['id', 'price', 'latitude', 'longitude', 'month', 
            'host_years', 'superhost', 'instant_bookable', 'is_entire_home',
            'number_of_reviews', 'review_scores_rating', 'amenities']

    # Ensure columns exist in all DFs
    all_dfs = [df_march, df_june]
    if not df_q4.empty: all_dfs.append(df_q4)

    for df in all_dfs:
        for c in cols:
            if c not in df.columns: df[c] = np.nan

    # Stack
    final_df = pd.concat([df[cols] for df in all_dfs], ignore_index=True)
    
    # Clean Prices
    final_df['price'] = final_df['price'].apply(clean_price)
    final_df.dropna(subset=['price'], inplace=True)
    
    # Fill NaNs
    final_df.fillna({
        'host_years': 0, 'superhost': 0, 'instant_bookable': 0, 
        'number_of_reviews': 0, 'is_entire_home': 1
    }, inplace=True)
    final_df['review_scores_rating'].fillna(final_df['review_scores_rating'].mean(), inplace=True)

    # --- 4. Amenities ---
    print("One-Hot Encoding Amenities (Scanning March, June, and Q4)...")
    final_df['amenities_parsed'] = final_df['amenities'].apply(parse_amenities)
    
    mlb = MultiLabelBinarizer()
    amenities_matrix = mlb.fit_transform(final_df['amenities_parsed'])
    
    clean_cols = [f"has_{c.lower().replace(' ', '_').replace('-', '_').replace('/', '')}" for c in mlb.classes_]
    amenities_df = pd.DataFrame(amenities_matrix, columns=clean_cols, index=final_df.index)
    
    # Drop rare amenities (less than 1%)
    min_count = 0.01 * len(final_df)
    amenities_df = amenities_df.loc[:, amenities_df.sum() > min_count]
    print(f"Kept {len(amenities_df.columns)} amenity features.")

    final_df = pd.concat([final_df, amenities_df], axis=1)
    final_df.drop(columns=['amenities', 'amenities_parsed'], inplace=True)

    # --- 5. Save ---
    output_file = "paris_complete_seasonal_dataset.csv"
    final_df.to_csv(output_file, index=False)
    
    print("="*40)
    print(f"SUCCESS! Saved to: {output_file}")
    print(f"Total Rows: {len(final_df)}")
    print(f"Months: {sorted(final_df['month'].unique())}") # Should show [3, 6, 10, 11, 12]
    print("="*40)



In [5]:
merge_datasets()

--- 1. Loading Datasets ---
Loaded March: (86064, 80)
Loaded June:  (84055, 80)
Loaded Q4:    (840, 54)

--- 2. Normalizing Columns ---

--- 3. Merging Everything ---
One-Hot Encoding Amenities (Scanning March, June, and Q4)...


/tmp/ipykernel_42951/314574220.py:140: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df['review_scores_rating'].fillna(final_df['review_scores_rating'].mean(), inplace=True)


Kept 146 amenity features.
SUCCESS! Saved to: paris_complete_seasonal_dataset.csv
Total Rows: 109618
Months: [np.int64(3), np.int64(6)]
